# 지식 증류 학습 · 학생 1개 × 교사 1개 · LOL-v2-real
**런타임 → 런타임 유형 변경 → GPU** 선택 후 위에서 아래로 실행하세요.
같은 학생 구조를 **일반 학습과 같은 initial.pt, 같은 설정·epoch**로 시작하고, 손실에만 교사 출력 항(β)을 더합니다.

| TEACHER_KIND | 교사 | 가능한 학생 |
|---|---|---|
| `base` | 일반 학습 Max+ (`gt_random_v1/maxplus_gt_s42/best.pt`) | mini, max |
| `strong` | Retinexformer (`teachers/teacher_v1/retinexformer_s42/best.pt`) | mini, max, maxplus |

준비물(이 노트북을 실행하는 사람의 Drive 기준 경로): 학생의 일반 학습 폴더(`initial.pt`, `run.json`, `summary.json`)와 교사 `best.pt`.
다른 팀원이 학습했다면 같은 경로로 복사해 둡니다. 교사는 학생과 **같은 random 패치**에서 매 배치 계산합니다.
β 후보는 0.1 / 0.3 / 1.0이며 검증으로 고르고, 고른 뒤에는 고정합니다.
설정(STUDENT·TEACHER_KIND·BETA)을 바꾸면 설정 셀을 다시 실행하세요. 각 실행 셀은 그 순간의 설정으로 경로·명령을 새로 만들고 준비 파일을 다시 검사합니다.

실행 코드는 GitHub 커밋 `1c2ad2128f75242559859b8ddc1eced21c3c88bf`로 고정됩니다.


In [ ]:
from pathlib import Path
STUDENT = "mini"        # mini / max / maxplus
TEACHER_KIND = "base"   # base: Max+ 일반 학습 / strong: Retinexformer
BETA = 0.3              # 후보 0.1 / 0.3 / 1.0
SEED = 42
EPOCHS = 120
EPOCHS_PER_SESSION = 120
BATCH_SIZE = 40
GT_EXPERIMENT = "gt_random_v1"
TEACHER_EXPERIMENT = "teacher_v1"
EXPERIMENT = "kd_random_v1"
BASE = Path("/content/drive/MyDrive/ondevice")
DATA_ZIP = BASE / "LOLv2.zip"
CODE_REF = '1c2ad2128f75242559859b8ddc1eced21c3c88bf'


## 1. Drive 연결 및 실행 환경 준비


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, os, shutil, subprocess, sys, uuid, zipfile
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택한 뒤 다시 실행하세요."
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def extract_checked(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f"잘못된 ZIP 경로: {name}")
        z.extractall(destination)

WORK = Path("/content/linet_gt")
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / f"source_{CODE_REF[:12]}"
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout",
                    "https://github.com/gidgogo/ondevice_project_v2.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REF], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual == CODE_REF, "코드 버전이 다릅니다. 새 런타임에서 다시 시작하세요."
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(REPO / "scripts"))
print("실행 코드:", actual)
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
CONFIG = REPO / "configs/colab_gt.json"

def kd_plan():
    # 실행 셀마다 호출: 그 순간의 설정 셀 값으로 경로·명령을 만들고 준비 파일을 검사한다.
    assert STUDENT in ("mini", "max", "maxplus") and TEACHER_KIND in ("base", "strong")
    assert not (STUDENT == "maxplus" and TEACHER_KIND == "base"), "Max+는 자기 자신(Max+ 일반 학습)을 교사로 쓰지 않습니다."
    assert BETA > 0 and isinstance(EPOCHS_PER_SESSION, int) and EPOCHS_PER_SESSION > 0
    gt_run = BASE / "runs" / GT_EXPERIMENT / f"{STUDENT}_gt_s{SEED}"
    teachers = {"base": ("maxplus", BASE / "runs" / GT_EXPERIMENT / "maxplus_gt_s42" / "best.pt"),
                "strong": ("retinexformer", BASE / "teachers" / TEACHER_EXPERIMENT / "retinexformer_s42" / "best.pt")}
    teacher_model, teacher_ckpt = teachers[TEACHER_KIND]
    initial = gt_run / "initial.pt"
    for path in (initial, gt_run / "run.json", teacher_ckpt):
        assert path.is_file(), f"준비 파일이 없습니다: {path}"
    base_config = json.loads(CONFIG.read_text())
    base_config.update(model=STUDENT, seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE)
    gt_config = json.loads((gt_run / "run.json").read_text())["config"]
    assert gt_config == base_config, f"일반 학습과 설정이 다릅니다. 같은 조건으로 비교해야 합니다.\nGT: {gt_config}\n지금: {base_config}"
    spec = {"model": teacher_model, "checkpoint": str(teacher_ckpt)}
    settings = [f"model={json.dumps(STUDENT)}", f"seed={SEED}", f"batch_size={BATCH_SIZE}",
                f"beta={BETA}", f"teacher={json.dumps(spec)}"]
    command = [sys.executable, "-u", str(REPO / "scripts/train.py"),
               "--data-root", str(DATA_ROOT), "--manifests", str(MANIFESTS), "--config", str(CONFIG)]
    return {"gt_run": gt_run, "initial": initial, "teacher_ckpt": teacher_ckpt, "settings": settings,
            "effective": dict(base_config, beta=BETA, teacher=spec), "command": command,
            "output": BASE / "runs" / EXPERIMENT / f"{STUDENT}_{TEACHER_KIND}kd_b{BETA}_s{SEED}"}


## 2. 데이터 준비
학습 620쌍·검증 69쌍의 파일 해시를 확인합니다. 시험 이미지는 읽지 않습니다.


In [ ]:
assert DATA_ZIP.is_file(), f"기존 데이터 ZIP을 Drive에 올려주세요: {DATA_ZIP}"
LOCAL_ZIP = WORK / "LOLv2.zip"
DATA_FOLDER = WORK / "data"
DATA_MARKER = DATA_FOLDER / ".complete"
EXPECTED_DATA_SHA256 = "298e00fece107b64fb54769ae8bc5175b4f84a3f7e8134e651a0a4b5eb61198d"
if not DATA_MARKER.exists():
    shutil.copy2(DATA_ZIP, LOCAL_ZIP)
    assert file_hash(LOCAL_ZIP) == EXPECTED_DATA_SHA256, "기존 프로젝트의 LOLv2.zip과 파일이 다릅니다."
    extract_checked(LOCAL_ZIP, DATA_FOLDER)
    DATA_MARKER.write_text(EXPECTED_DATA_SHA256)
roots = [p for p in DATA_FOLDER.rglob("Real_captured") if (p / "Train/Low").is_dir()]
assert len(roots) == 1, f"Real_captured 폴더를 하나 찾을 수 있어야 합니다: {roots}"
DATA_ROOT = roots[0]
MANIFESTS = REPO / "manifests/lolv2_real"
verified = set()
for split, count in [("train", 620), ("validation", 69)]:
    manifest = json.loads((MANIFESTS / f"{split}.json").read_text())
    assert manifest["split"] == split and len(manifest["pairs"]) == count
    for row in manifest["pairs"]:
        for key in ("low", "high"):
            path = DATA_ROOT / row[key]
            if path not in verified:
                assert file_hash(path) == row[f"{key}_sha256"], f"데이터 불일치: {path}"
                verified.add(path)
    print(split, count, "쌍 확인 완료")


## 3. 짧은 실행 점검
2배치 학습 + 검증 69장. 교사 추론이 포함된 시간을 확인합니다.


In [ ]:
plan = kd_plan()
print("학생:", STUDENT, "· 교사:", TEACHER_KIND, plan["teacher_ckpt"], file_hash(plan["teacher_ckpt"])[:12], "· β:", BETA)
print("일반 학습 초기값:", plan["initial"], file_hash(plan["initial"])[:12])
pilot_dir = BASE / "pilots" / EXPERIMENT / f"{STUDENT}_{TEACHER_KIND}_{uuid.uuid4().hex[:8]}"
subprocess.run(plan["command"] + ["--output", str(pilot_dir), "--max-train-batches", "2",
                                  "--init-checkpoint", str(plan["initial"]), "--set", *plan["settings"], "epochs=1"],
               check=True, cwd=REPO)
print("점검 성공")


## 4. 본 학습 / 이어서 학습
처음에는 일반 학습의 initial.pt에서 시작하고, 이후에는 마지막 저장 epoch에서 이어갑니다.


In [ ]:
plan = kd_plan()
OUTPUT = plan["output"]
resumable = any((OUTPUT / name).exists() for name in ("last.pt", "last.prev.pt"))
if OUTPUT.exists() and any(OUTPUT.iterdir()) and not resumable:
    raise RuntimeError("복구할 checkpoint가 없습니다. EXPERIMENT를 새 이름으로 바꾸세요.")
if (OUTPUT / "run.json").exists():
    assert json.loads((OUTPUT / "run.json").read_text())["config"] == plan["effective"], "기존 실험과 설정이 다릅니다. 새 EXPERIMENT 이름을 사용하세요."
cmd = plan["command"] + ["--output", str(OUTPUT), "--epochs-per-session", str(EPOCHS_PER_SESSION)]
cmd += ["--resume"] if resumable else ["--init-checkpoint", str(plan["initial"])]
print("실행 설정:", plan["settings"], "→", OUTPUT)
subprocess.run(cmd + ["--set", *plan["settings"], f"epochs={EPOCHS}"], check=True, cwd=REPO)
progress = json.loads((OUTPUT / "progress.json").read_text())
print(f"저장 완료: {progress['completed_epochs']}/{progress['target_epochs']} epoch")


## 5. 일반 학습과 비교 (검증 69장)
같은 학생·같은 초기값·같은 epoch에서 증류만 다릅니다. 시험 분할은 쓰지 않습니다.


In [ ]:
plan = kd_plan()
kd = json.loads((plan["output"] / "summary.json").read_text())
print("증류:", kd["best"])
gt_summary = plan["gt_run"] / "summary.json"
if gt_summary.exists() and json.loads(gt_summary.read_text()).get("training_complete"):
    gt = json.loads(gt_summary.read_text())["best"]
    print("일반 학습:", gt)
    if kd.get("training_complete"):
        print(f"증류 - 일반 학습: PSNR {kd['best']['psnr'] - gt['psnr']:+.3f} dB, SSIM {kd['best']['ssim'] - gt['ssim']:+.4f}")
else:
    print("일반 학습이 아직 완료되지 않았거나 summary.json이 없습니다.")
